In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.io.shapereader as shpreader
import pycountry


### File paths

In [ ]:
production_fn = '../../resources/steel_production.csv'

### Workflow

In [ ]:
def country_to_iso_a2(country_name):
    try:
        return pycountry.countries.lookup(country_name).alpha_2
    except LookupError:
        return None

In [ ]:
# %%
# Load OWID steel data: https://ourworldindata.org/explorers/minerals?tab=map&Mineral=Iron+ore&Metric=Production&Type=Mine%2C+crude+ore&Share+of+global=false&country=OWID_WRL~AUS~CHL~CHN~USA
df = pd.read_csv("../../data/owid-steel/steel-production.csv")

# Rename columns for easier access
df.rename(columns={
    'Entity': 'Country',
    'Year': 'Year',
    'production|Steel|Processing, crude|tonnes': 'SteelProduction',
}, inplace=True)

# Keep only rows with available steel production data
df = df.dropna(subset=['SteelProduction'])

# Get latest year per country
df_latest = df.sort_values('Year').groupby('Country', as_index=False).last()

# Drop data that is older than 2010 and drop Country 'World'
df_latest = df_latest[df_latest['Country'] != 'World']
df_latest = df_latest[df_latest['Year'] >= 2010]

In [ ]:
# Map problematic country names to official names for ISO_A2 conversion
country_name_corrections = {
    "Democratic Republic of Congo": "Congo, The Democratic Republic of the",
    "Kosovo": "Republic of Kosovo",  # pycountry not supported
    "Russia": "Russian Federation",
    "Turkey": "Türkiye"
}

df_latest['Country'] = df_latest['Country'].replace(country_name_corrections)
df_latest['ISO_A2'] = df_latest['Country'].apply(country_to_iso_a2)

# Check for any remaining missing ISO_A2 codes
missing_iso_a2 = df_latest[df_latest['ISO_A2'].isna()]
print(f"Countries with missing ISO_A2 codes after correction: {missing_iso_a2['Country'].unique()}")

In [ ]:
print(f"Total steel production in latest year (Mt): {df_latest.SteelProduction.sum()/1e6:.2f} Mt")

In [ ]:
# Sort values
df_latest.sort_values('SteelProduction', ascending=False)

In [ ]:
# Use cartopy to get natural earth countries shapefile
shapename = 'admin_0_countries'
reader = shpreader.natural_earth(resolution='110m',
                                 category='cultural',
                                 name=shapename)

world = gpd.read_file(reader)

# Fix missing ISO_A2 codes in world
world.loc[world['ISO_A2'] == '-99', 'ISO_A2'] = world.loc[world['ISO_A2'] == '-99', 'ADMIN'].apply(country_to_iso_a2)

# Merge country names — use left join to preserve geometry
merged = world.merge(df_latest, how='left', left_on='ISO_A2', right_on='ISO_A2')

In [ ]:
# Convert to megatonnes
merged['SteelProductionMt'] = merged['SteelProduction'].divide(1e6) # tonnes to Mt

In [ ]:
# Ensure ISO_A2 and IronOreProductionGt exist
filtered_df = merged[
    (merged['SteelProductionMt'] > 0) & 
    (merged['ISO_A2'].notna())
][['ISO_A2', 'SteelProductionMt', 'ADMIN']]
# Set ISO_A2 as index
production = filtered_df.set_index('ISO_A2')

# Display result
production.to_csv(production_fn)

In [ ]:

# Define Gt bins and labels
bins = [0, 1, 10, 50, 100, 500, float('inf')]
labels = ['0–1', '1–10', '10–50', '50–100', '100–500', '500+']


# Bin into categories
merged['ProductionCategory'] = pd.cut(
    merged['SteelProductionMt'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# %%
# Plot
fig, ax = plt.subplots(figsize=(10, 7))

# Base world map
world.plot(ax=ax, color='lightgrey', edgecolor='white')

# Plot with categorical Gt bins
merged.dropna(subset=['ProductionCategory']).plot(
    ax=ax,
    column='ProductionCategory',
    cmap='Blues',
    legend=True,
    legend_kwds={'title': "Steel Production (Mt)"},
    missing_kwds={"color": "lightgrey"}
)

ax.set_title("Steel Production by Country")
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
production.sort_values('SteelProductionMt', ascending=False).iloc[0:20,:]
